In [1]:
#!pip install dtreeviz graphviz ipykernel cairosvg

In [2]:
import dtreeviz
import graphviz

In [3]:
import os

graphviz_bin = r'C:/Program Files/Graphviz/bin'

os.environ['PATH'] = graphviz_bin + os.pathsep + os.environ['PATH']

In [4]:
import matplotlib as mpl
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

mpl.rcParams['svg.fonttype'] = 'none'

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import warnings

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

plt.style.use("ggplot")

In [6]:
df = pd.read_csv('jeju_bus.csv')

display(df.head())

print('shape:', df.shape)

print('전체 결측치 개수:', df.isnull().sum().sum())

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,next_station,next_latitude,next_longitude,next_arrive_time
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,제대마을,33.457724,126.554014,24
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,제대아파트,33.458783,126.557353,36
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,제주대학교,33.459893,126.561624,40
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,제주여자중고등학교(아라방면),33.484860,126.542928,42
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,은남동,33.485822,126.490897,64


shape: (210457, 14)
전체 결측치 개수: 0


In [7]:
df_model = df.copy()

df_model['original_index'] = df_model.index
target_col = 'next_arrive_time'

In [8]:
df_model['date'] = pd.to_datetime(df_model['date'])

df_model['day'] = df_model['date'].dt.day
df_model['dayofweek'] = df_model['date'].dt.dayofweek

df_model['now_hour'] = df_model['now_arrive_time'].astype(str).str.extract(r'(\d+)').astype(float)

df_model[['date', 'day', 'dayofweek', 'now_arrive_time', 'now_hour']].head()

,date,day,dayofweek,now_arrive_time,now_hour
0,2019-10-15,15,1,06시,6.0
1,2019-10-15,15,1,06시,6.0
2,2019-10-15,15,1,06시,6.0
3,2019-10-15,15,1,06시,6.0
4,2019-10-15,15,1,07시,7.0


df_model['station_segment'] = (
    df_model['now_station'].astype(str) + ' → ' + df_model['next_station'].astype(str)
)
df_model.head()

print('station_segment 고유값 개수:', df_model['station_segment'].nunique())

In [9]:
def calculate_distance_km(lat1, lon1, lat2, lon2):
    # 위도/경도 사이의 거리를 km 단위로 계산!
    # 1: 현재 내 정류징
    # 2: 각 주요 지점
    earth_radius_km = 6371

    # radius: 반지름, 반
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    # 거리 계산 공식
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2        
    )
    c = 2 * np.arcsin(np.sqrt(a))
    return earth_radius_km * c

In [10]:
reference_points = {
    'up'    : (33.506286, 126.490312),
    'down'  : (33.246742, 126.562387),
    'right' : (33.493521, 126.895326),
    'center': (33.379724, 126.545315)
}

In [11]:
def assign_region_info(data, lat_col, lon_col):
    up_lat, up_lon = reference_points['up']
    down_lat, down_lon = reference_points['down']
    right_lat, right_lon = reference_points['right']
    center_lat, center_lon = reference_points['center']

    distance_to_up = calculate_distance_km(data[lat_col], data[lon_col], up_lat, up_lon)
    distance_to_down = calculate_distance_km(data[lat_col], data[lon_col], down_lat, down_lon)
    distance_to_right = calculate_distance_km(data[lat_col], data[lon_col], right_lat, right_lon)
    distance_to_center = calculate_distance_km(data[lat_col], data[lon_col], center_lat, center_lon)

    distance_table = pd.DataFrame({
        'up': distance_to_up,
        'down': distance_to_down,
        'right': distance_to_right,
        'center': distance_to_center        
    }, index = data.index)

    nearest_region = distance_table.idxmin(axis = 1)

    result = pd.DataFrame({
        'dist_name': nearest_region,
        'dist_to_up': distance_to_up,
        'dist_to_down': distance_to_down,
        'dist_to_right': distance_to_right,
        'dist_to_center': distance_to_center
    }, index = data.index)

    return result

In [12]:
now_region_info = assign_region_info(
    df_model,
    'now_latitude',
    'now_longitude'    
)

now_region_info.head()

,dist_name,dist_to_up,dist_to_down,dist_to_right,dist_to_center
0,up,7.962505,23.319056,32.135034,8.532122
1,up,8.003869,23.473015,31.905930,8.710701
2,up,8.158338,23.582519,31.583874,8.861671
3,up,5.774762,25.961685,32.635035,11.118256
4,up,2.332803,27.295447,37.141639,12.673969


In [13]:
next_region_info = assign_region_info(
    df_model,
    'next_latitude',
    'next_longitude'    
)

next_region_info.head()

,dist_name,dist_to_up,dist_to_down,dist_to_right,dist_to_center
0,up,8.003869,23.473015,31.905930,8.710701
1,up,8.158338,23.582519,31.583874,8.861671
2,up,8.387595,23.701416,31.175511,9.041978
3,up,5.429627,26.539110,32.693959,11.692688
4,up,2.276139,27.400941,37.514443,12.832868


In [14]:
df_model['now_dist_name'] = now_region_info['dist_name']
df_model['now_dist_to_up'] = now_region_info['dist_to_up']
df_model['now_dist_to_down'] = now_region_info['dist_to_down']
df_model['now_dist_to_right'] = now_region_info['dist_to_right']
df_model['now_dist_to_center'] = now_region_info['dist_to_center']

df_model.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,...,next_arrive_time,original_index,day,dayofweek,now_hour,now_dist_name,now_dist_to_up,now_dist_to_down,now_dist_to_right,now_dist_to_center
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,...,24,0,15,1,6.0,up,7.962505,23.319056,32.135034,8.532122
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,...,36,1,15,1,6.0,up,8.003869,23.473015,31.905930,8.710701
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,...,40,2,15,1,6.0,up,8.158338,23.582519,31.583874,8.861671
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,...,42,3,15,1,6.0,up,5.774762,25.961685,32.635035,11.118256
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,...,64,4,15,1,7.0,up,2.332803,27.295447,37.141639,12.673969


In [15]:
df_model['next_dist_name'] = next_region_info['dist_name']
df_model['next_dist_to_up'] = next_region_info['dist_to_up']
df_model['next_dist_to_down'] = next_region_info['dist_to_down']
df_model['next_dist_to_right'] = next_region_info['dist_to_right']
df_model['next_dist_to_center'] = next_region_info['dist_to_center']

df_model.head()

,id,date,route_id,vh_id,route_nm,now_latitude,now_longitude,now_station,now_arrive_time,distance,...,now_dist_name,now_dist_to_up,now_dist_to_down,now_dist_to_right,now_dist_to_center,next_dist_name,next_dist_to_up,next_dist_to_down,next_dist_to_right,next_dist_to_center
0,0,2019-10-15,405136001,7997025,360-1,33.456267,126.551750,제주대학교입구,06시,266.0,...,up,7.962505,23.319056,32.135034,8.532122,up,8.003869,23.473015,31.905930,8.710701
1,1,2019-10-15,405136001,7997025,360-1,33.457724,126.554014,제대마을,06시,333.0,...,up,8.003869,23.473015,31.905930,8.710701,up,8.158338,23.582519,31.583874,8.861671
2,2,2019-10-15,405136001,7997025,360-1,33.458783,126.557353,제대아파트,06시,415.0,...,up,8.158338,23.582519,31.583874,8.861671,up,8.387595,23.701416,31.175511,9.041978
3,3,2019-10-15,405136001,7997025,360-1,33.479705,126.543811,남국원(아라방면),06시,578.0,...,up,5.774762,25.961685,32.635035,11.118256,up,5.429627,26.539110,32.693959,11.692688
4,4,2019-10-15,405136001,7997025,360-1,33.485662,126.494923,도호동,07시,374.0,...,up,2.332803,27.295447,37.141639,12.673969,up,2.276139,27.400941,37.514443,12.832868


In [16]:
df_model['dist_segment_name'] = (
    df_model['now_dist_name'].astype(str)
    + ' → '
    + df_model['next_dist_name'].astype(str)    
)

print('dist_segment_name 고유값 개수: ', df_model['dist_segment_name'].nunique())
df_model['dist_segment_name'].value_counts()

dist_segment_name 고유값 개수:  12


dist_segment_name
up → up            121131
down → down         48377
right → right       34032
center → center      5512
center → up           321
right → up            276
up → center           207
down → right          144
up → right            137
right → down          134
down → center          95
center → down          91
Name: count, dtype: int64

In [17]:
df_model.columns.tolist()

['id',
 'date',
 'route_id',
 'vh_id',
 'route_nm',
 'now_latitude',
 'now_longitude',
 'now_station',
 'now_arrive_time',
 'distance',
 'next_station',
 'next_latitude',
 'next_longitude',
 'next_arrive_time',
 'original_index',
 'day',
 'dayofweek',
 'now_hour',
 'now_dist_name',
 'now_dist_to_up',
 'now_dist_to_down',
 'now_dist_to_right',
 'now_dist_to_center',
 'next_dist_name',
 'next_dist_to_up',
 'next_dist_to_down',
 'next_dist_to_right',
 'next_dist_to_center',
 'dist_segment_name']

In [18]:
selected_numeric_features = [
     'distance', 
     'day',  
     'dayofweek',
     'now_hour',
     'now_dist_to_up',
     'now_dist_to_down',
     'now_dist_to_right',
     'now_dist_to_center',
     'next_dist_to_up',
     'next_dist_to_down',
     'next_dist_to_right',
     'next_dist_to_center'
]

In [19]:
selected_categorical_features = [
     'route_nm',
     'now_station',
     'next_station',
     'dist_segment_name'
]

In [20]:
selected_features = selected_numeric_features + selected_categorical_features
selected_features

['distance',
 'day',
 'dayofweek',
 'now_hour',
 'now_dist_to_up',
 'now_dist_to_down',
 'now_dist_to_right',
 'now_dist_to_center',
 'next_dist_to_up',
 'next_dist_to_down',
 'next_dist_to_right',
 'next_dist_to_center',
 'route_nm',
 'now_station',
 'next_station',
 'dist_segment_name']

In [21]:
upper_1pct = df_model[target_col].quantile(0.99)

upper_1pct

np.float64(340.0)

In [22]:
df_model_selected = df_model [ df_model[target_col] <= upper_1pct ].copy()

In [23]:
X = df_model_selected[selected_features]

In [24]:
y = df_model_selected[target_col]

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
                                         X, y, 
                                         test_size = 0.2,
                                         random_state = 42
                                    )

In [26]:
preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown = 'ignore'), selected_categorical_features)
        ], remainder = 'passthrough'
    )

In [27]:
def evaluate_regression_model(model_name, description, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f'[{model_name}] MAE : {mae:.4f} | RMSE : {rmse:.4f}')

    return {'model_name': model_name, 'description': description,
            'MAE': mae, 'RMSE' : rmse
    }

In [28]:
def train_xgb_pipeline(model_name1, description1, xgb_params):
    xgb_model = XGBRegressor(
        objective = 'reg:squarederror',
        random_state = 42,
        n_jobs = -1,# 가용 CPU 최대 사용
        **xgb_params
    )

    model_pipeline = Pipeline(steps=[
                             ('preprocessor', preprocessor),
                             ('model', xgb_model)
                     ])

    model_pipeline.fit(X_train, y_train)
    y_pred1 = model_pipeline.predict(X_test)

    evaluation = evaluate_regression_model(
                                model_name=model_name1,
                                description = description1,
                                y_true = y_test,
                                y_pred = y_pred1
                                )
    return{"model_name": model_name1, 'pipeline': model_pipeline,
          'ypred':y_pred1, 'evaluation': evaluation}                             
    

In [29]:
shallow_result = train_xgb_pipeline(
                model_name1 = 'shallow_xgb',
                description1 = '3층 트리 많이!!',
                xgb_params = {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 3}
            )

[shallow_xgb] MAE : 23.1969 | RMSE : 31.6257


In [30]:
deep_result = train_xgb_pipeline(
    model_name1 = 'deep_xgb',
    description1 = '깊은트리 중요하다',
    xgb_params = {
            'n_estimators': 200,
            'learning_rate': 0.05,
            'max_depth': 8
            }
    )

[deep_xgb] MAE : 19.7910 | RMSE : 28.3630


In [31]:
slow_deep_result = train_xgb_pipeline(
    model_name1 = 'slow_deep_xgb',
    description1 = '깊은트리 천천히 많이 학습 중요하다',
    xgb_params = {
            'n_estimators': 500,
            'learning_rate': 0.03,
            'max_depth': 8
            }
    )

[slow_deep_xgb] MAE : 19.5464 | RMSE : 28.1803


In [32]:
regularized_result = train_xgb_pipeline(
    model_name1 = 'regularized_xgb',
    description1 = '매번 다른 데이터 랜덤하게 줘서 특정행 컬럼에 의지하지 않게 조금씩 여러번 수정',
    xgb_params = {
        'n_estimators': 500,
        'learning_rate': 0.05,
        'max_depth': 8,
        'sumsample': 0.8,# 트리마다 행을 랜덤하게 배정
        'colsample_bytree': 0.8,# 트리마다 열도 랜덤하게 배정
        'reg_alpha': 0.01,#L1 규제: 트리 너무 깊게X
        'reg_lambda': 1.5#L2 규제: 학습 속도 천천히        
    }
)

[regularized_xgb] MAE : 19.2083 | RMSE : 27.9174


In [33]:
fold_df = pd.DataFrame({
    'distance':        [0.5, 1.0, 0.7, 1.5, 0.3],
    'now_hour':        [8,   7,   14,  18,  10 ],
    'next_arrive_time':[90,  150, 100, 240, 60 ]
})

In [34]:
fold_df

,distance,now_hour,next_arrive_time
0,0.5,8,90
1,1.0,7,150
2,0.7,14,100
3,1.5,18,240
4,0.3,10,60


In [35]:
fold_X = fold_df[['distance', 'now_hour']]

fold_X

,distance,now_hour
0,0.5,8
1,1.0,7
2,0.7,14
3,1.5,18
4,0.3,10


In [36]:
fold_y = fold_df['next_arrive_time']

fold_y

0     90
1    150
2    100
3    240
4     60
Name: next_arrive_time, dtype: int64

In [37]:
from sklearn.model_selection import KFold

In [38]:
kf = KFold(n_splits = 5, shuffle = False)

In [39]:
# num = 1
# mae_scores = []

# for train_idx, valid_idx in kf.split(fold_X):
#     print("=" * 30)
#     print(f"num={num}")
#     print(f"train_idx={train_idx}"); print(f"valid_idx={valid_idx}")

#     X_train = fold_X.iloc[train_idx]
#     y_train = fold_y.iloc[train_idx]
#     print(f"X_train={X_train}"); print(f"y_train={y_train}")

#     X_valid = fold_X.iloc[valid_idx]
#     y_valid = fold_y.iloc[valid_idx]
#     print(f"X_valid={X_valid}"); print(f"y_valid={y_valid}")

#     model = XGBRegressor()
#     model.fit(X_train, y_train)
#     y_pred = model.predict(X_valid)

#     mae = mean_absolute_error(y_valid, y_pred)
#     print(f"mae={mae}")
#     mae_scores.append(mae)
#     num = num + 1

In [40]:
# mae_scores

NameError: name 'mae_scores' is not defined

In [ ]:
# np.mean(mae_scores)

In [41]:
param_di = {
    'model__max_depth':[8, 9, 10],# 트리 깊이
    'model__learning_rate': [0.03, 0.05, 0.1],# 학습률(각 트리 보정 반영 비율)
    'model__n_estimators': [400, 500],# 트리 갯수
    'model__subsample': [0.8, 0.9, 1.0],# 트리마다 행을 랜덤하게 배정
    'model__colsample_bytree': [0.8, 0.9, 1.0],# 트리마다 열도 랜덤하게 배정
}

scoring = 'neg_mean_abosolute_error'

In [42]:
search_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=1        
    ))    
])

In [43]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator = search_model,
    param_distributions = param_di,
    n_iter = 30,
    scoring = 'neg_mean_absolute_error',
    cv = 5,
    random_state = 42,
    n_jobs = -1,
    verbose = 1
)

In [44]:
import time

strat_time = time.time()
random_search.fit(X_train, y_train)
elasped = time.time() - strat_time

print(f'소요시간: {elasped}초')

Fitting 5 folds for each of 30 candidates, totalling 150 fits
소요시간: 752.5420198440552초


In [46]:
random_search.best_params_

{'model__subsample': 0.9,
 'model__n_estimators': 500,
 'model__max_depth': 9,
 'model__learning_rate': 0.1,
 'model__colsample_bytree': 0.9}

In [47]:
random_search.best_score_

np.float64(-18.77615394592285)

In [48]:
best_model = random_search.best_estimator_

In [54]:
best_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [55]:
y_pred_best = best_model.predict(X_test)

y_pred_best

array([200.26898 ,  59.078136, 110.52966 , ...,  64.570496,  35.924217,
        44.09762 ], shape=(41671,), dtype=float32)

In [56]:
evaluate_regression_model(model_name='best_tunned_xgb',
                          description = '최적의 모델',
                          y_true = y_test,
                          y_pred = y_pred_best
                         )

[best_tunned_xgb] MAE : 18.7165 | RMSE : 27.9148


{'model_name': 'best_tunned_xgb',
 'description': '최적의 모델',
 'MAE': 18.71653175354004,
 'RMSE': np.float64(27.914768742815042)}

In [58]:
best_preprocessor = best_model.named_steps['preprocessor']# 전처리기

best_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{fea

In [61]:
best_xgb_model = best_model.named_steps['model']# 실제 XGBoost 모델

print(f'best_xgb_model = {best_xgb_model}')

best_xgb_model = XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=9,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=1, num_parallel_tree=None, ...)


In [63]:
# text: 이름 앞에 remainder
transformed_feature_names = best_preprocessor.get_feature_names_out()

transformed_feature_names

array(['cat__route_nm_201-11', 'cat__route_nm_201-12',
       'cat__route_nm_201-13', 'cat__route_nm_201-14',
       'cat__route_nm_201-15', 'cat__route_nm_201-16',
       'cat__route_nm_201-17', 'cat__route_nm_201-18',
       'cat__route_nm_201-21', 'cat__route_nm_201-22',
       'cat__route_nm_201-24', 'cat__route_nm_201-26',
       'cat__route_nm_201-27', 'cat__route_nm_281-1',
       'cat__route_nm_281-2', 'cat__route_nm_360-1',
       'cat__route_nm_360-12', 'cat__route_nm_360-2',
       'cat__route_nm_360-7', 'cat__route_nm_365-21',
       'cat__route_nm_365-22', 'cat__now_station_911의원',
       'cat__now_station_LH아파트', 'cat__now_station_가마초등학교',
       'cat__now_station_가흥동', 'cat__now_station_거로 입구',
       'cat__now_station_견월교', 'cat__now_station_계룡동',
       'cat__now_station_고도농원', 'cat__now_station_고래왓',
       'cat__now_station_고망난돌입구', 'cat__now_station_고산동산(광양방면)',
       'cat__now_station_고산동산(아라방면)', 'cat__now_station_고성리 구 성산농협',
       'cat__now_station_고성리 성산농협', 

In [65]:
best_xgb_model.feature_importances_

array([1.07486499e-04, 1.20048207e-04, 1.01617297e-04, 1.54137131e-04,
       1.66869198e-04, 1.07575601e-04, 1.08487999e-04, 1.41891040e-04,
       1.40668868e-04, 1.17957126e-04, 1.14932518e-04, 1.38622883e-04,
       1.22191908e-04, 2.67385476e-04, 3.55350581e-04, 2.60404806e-04,
       2.02683543e-04, 1.87311918e-04, 2.02857918e-04, 2.11972874e-04,
       7.82589603e-04, 4.38283256e-04, 8.63071473e-04, 1.71967076e-05,
       1.55603248e-05, 2.22578790e-04, 3.89940920e-04, 3.61396415e-05,
       6.39823440e-04, 1.45321875e-03, 1.62086042e-04, 1.10195216e-03,
       3.22445238e-04, 4.62259119e-03, 1.66905098e-04, 6.37087971e-03,
       2.00723531e-04, 2.99412233e-04, 1.40223093e-03, 1.78776536e-04,
       9.03188484e-04, 3.63204454e-04, 1.25803263e-03, 5.07939083e-04,
       3.03172768e-04, 1.11079476e-04, 2.29659145e-05, 7.28895902e-05,
       3.84974526e-04, 7.97025918e-04, 1.06194100e-04, 4.53604189e-05,
       5.45627190e-05, 4.24612546e-04, 2.18654968e-04, 9.95877053e-05,
      

In [68]:
feature_importance_df = pd.DataFrame({
    'feature': transformed_feature_names,
    'importance': best_xgb_model.feature_importances_
})

In [74]:
feature_importance_df.sort_values(
                by='importance', ascending = False
        ).reset_index(drop = True).head(30)

,feature,importance
0,cat__next_station_제주중앙여자고등학교(아라방면),0.084002
1,remainder__next_dist_to_up,0.040678
2,cat__next_station_제주중앙여자고등학교(광양방면),0.039622
3,cat__now_station_성산환승정류장(고성리 회전교차로),0.038387
4,cat__next_station_성판악,0.036422
5,remainder__distance,0.030773
6,cat__dist_segment_name_up → up,0.028900
7,cat__now_station_원노형,0.028139
8,cat__now_station_농협 하나로마트,0.023696
9,cat__now_station_월성 마을,0.018470


In [85]:
selected_categorical_features

['route_nm', 'now_station', 'next_station', 'dist_segment_name']

In [86]:
categorical_features = selected_categorical_features

categorical_features

['route_nm', 'now_station', 'next_station', 'dist_segment_name']

In [94]:
for transformed_feature_name in feature_importance_df['feature']:
    print(f'transformed_feature_name : {transformed_feature_name}')
    name = transformed_feature_name.replace('cat__', '', 1)
    print(f'name = {name}')
    for col in categorical_features:
        print(f'col = {col}')
        if name.startswith(col+'_'):
            print(f'name이 {col}_으로 시작!!')
            print(f'컬럼명은 {col}로 수정!!')
            break
            return col
        return name
    print('=' * 30)

transformed_feature_name : cat__route_nm_201-11
name = route_nm_201-11
col = route_nm
name이 route_nm_으로 시작!!
컬럼명은 route_nm로 수정!!
transformed_feature_name : cat__route_nm_201-12
name = route_nm_201-12
col = route_nm
name이 route_nm_으로 시작!!
컬럼명은 route_nm로 수정!!
transformed_feature_name : cat__route_nm_201-13
name = route_nm_201-13
col = route_nm
name이 route_nm_으로 시작!!
컬럼명은 route_nm로 수정!!
transformed_feature_name : cat__route_nm_201-14
name = route_nm_201-14
col = route_nm
name이 route_nm_으로 시작!!
컬럼명은 route_nm로 수정!!
transformed_feature_name : cat__route_nm_201-15
name = route_nm_201-15
col = route_nm
name이 route_nm_으로 시작!!
컬럼명은 route_nm로 수정!!
transformed_feature_name : cat__route_nm_201-16
name = route_nm_201-16
col = route_nm
name이 route_nm_으로 시작!!
컬럼명은 route_nm로 수정!!
transformed_feature_name : cat__route_nm_201-17
name = route_nm_201-17
col = route_nm
name이 route_nm_으로 시작!!
컬럼명은 route_nm로 수정!!
transformed_feature_name : cat__route_nm_201-18
name = route_nm_201-18
col = route_nm
name이 route

In [99]:
def get_original_feature_names(
    transformed_name, categorical_features
):
    print('=' * 30)
    print(f'transformed_name = {transformed_name}')
    print(f'categorical_features = {categorical_features}')
    name = transformed_name.replace('cat__' ,'', 1)
    print(f'name = {name}')
    for col in categorical_features:
        print(f'col = {col}')
        if name == col or name.startswith(col + '_'):
            return col
    return name

In [103]:
feature_importance_df['original_feature'] = feature_importance_df['feature'].apply(
    lambda x: get_original_feature_names(x,selected_categorical_features)            
)

transformed_name = cat__route_nm_201-11
categorical_features = ['route_nm', 'now_station', 'next_station', 'dist_segment_name']
name = route_nm_201-11
col = route_nm
transformed_name = cat__route_nm_201-12
categorical_features = ['route_nm', 'now_station', 'next_station', 'dist_segment_name']
name = route_nm_201-12
col = route_nm
transformed_name = cat__route_nm_201-13
categorical_features = ['route_nm', 'now_station', 'next_station', 'dist_segment_name']
name = route_nm_201-13
col = route_nm
transformed_name = cat__route_nm_201-14
categorical_features = ['route_nm', 'now_station', 'next_station', 'dist_segment_name']
name = route_nm_201-14
col = route_nm
transformed_name = cat__route_nm_201-15
categorical_features = ['route_nm', 'now_station', 'next_station', 'dist_segment_name']
name = route_nm_201-15
col = route_nm
transformed_name = cat__route_nm_201-16
categorical_features = ['route_nm', 'now_station', 'next_station', 'dist_segment_name']
name = route_nm_201-16
col = route_nm
tran

In [106]:
feature_importance_df

,feature,importance,original_feature
0,cat__route_nm_201-11,0.000107,route_nm
1,cat__route_nm_201-12,0.000120,route_nm
2,cat__route_nm_201-13,0.000102,route_nm
3,cat__route_nm_201-14,0.000154,route_nm
4,cat__route_nm_201-15,0.000167,route_nm
...,...,...,...
738,remainder__now_dist_to_center,0.003852,remainder__now_dist_to_center
739,remainder__next_dist_to_up,0.040678,remainder__next_dist_to_up
740,remainder__next_dist_to_down,0.017995,remainder__next_dist_to_down
741,remainder__next_dist_to_right,0.013281,remainder__next_dist_to_right


In [116]:
grouped_importance_df = (feature_importance_df.groupby('original_feature', as_index = False)['importance'].sum() \
                            .sort_values(by = 'importance', ascending = False).reset_index(drop=True)
                        )

grouped_importance_df

,original_feature,importance
0,next_station,0.409737
1,now_station,0.409255
2,remainder__next_dist_to_up,0.040678
3,dist_segment_name,0.032753
4,remainder__distance,0.030773
5,remainder__next_dist_to_down,0.017995
6,remainder__now_dist_to_up,0.013794
7,remainder__next_dist_to_right,0.013281
8,remainder__next_dist_to_center,0.011679
9,remainder__now_dist_to_down,0.006668


In [31]:
# from sklearn.model_selection import RandomizedSearchCV

# random_search = RandomizedSearchCV(
#     estimator = search_model,
#     param_distributions = param_di,
#     n_iter = 25,# 시도할 조합 수(실행이 오래 걸리면 10으로 줄일 수 있음)
#     scoring = 'neg_mean_absolute_error',# 음수 MAE(클수록 좋음)
#     cv = 5,# train을 5개로 나눠 내부 교차검증
#     random_state = 42,
#     n_jobs = -1,
#     verbose = 1# 진행 상황 출력
# )    

In [32]:
# random_search.fit(X_train, y_train)